# Qwen3.5-4B crystallization search + calibration (single-run Colab)

Этот notebook рассчитан на **один основной дорогой прогон**. Он делает всё в одном запуске:

1. снимает **all-layer geometry profiles** по anchor cases;
2. ищет **reference layers именно под `Qwen/Qwen3.5-4B`**;
3. ищет **crystallization thresholds** для `flat / template / mature`;
4. считает **generation calibration + policy simulation**;
5. сохраняет один JSON и один Markdown report.

Важно: это **не** legacy detector threshold sweep из `scripts/calibrate_qwen_anchor_thresholds.py`. Здесь ищутся именно **слои и пороги кристаллизации** под новую модель.


> Рекомендуемый runtime: **GPU (T4 / L4 / A100)**.
>
> Модель `Qwen/Qwen3.5-4B` требует актуальный `transformers`, поэтому notebook ставит `transformers` с `main`.
>
> Артефакты сохраняются с префиксом `qwen35_4b_`, чтобы не затирать старые результаты.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q --upgrade pip
!pip install -q torch torchvision pillow accelerate einops pytest numpy nbformat "transformers @ git+https://github.com/huggingface/transformers.git@main"


In [ ]:
%cd /content
import os
import subprocess
from pathlib import Path

if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
else:
    %cd /content/ABPT
    !git pull --ff-only

%cd /content/ABPT
print('HEAD =', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
import json
from pathlib import Path

MODEL_NAME = 'Qwen/Qwen3.5-4B'
DEVICE = 'cuda'
MAX_LENGTH = 160
MAX_NEW_TOKENS = 120
CAL_JSON = '/content/ABPT/archive/qwen35_4b_geometry_generation_calibration.json'
CAL_MD = '/content/ABPT/docs/research/qwen35_4b_geometry_generation_calibration.md'

for path in [CAL_JSON, CAL_MD]:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

print('MODEL_NAME =', MODEL_NAME)
print('DEVICE =', DEVICE)
print('MAX_LENGTH =', MAX_LENGTH)
print('MAX_NEW_TOKENS =', MAX_NEW_TOKENS)


## Main run: all-layer geometry -> searched reference layers -> searched thresholds -> calibration

Это основной дорогой шаг. Именно его и стоит запускать в Colab первым.


In [ ]:
!python scripts/run_qwen_geometry_generation_calibration.py --model "$MODEL_NAME" --device "$DEVICE" --max_length $MAX_LENGTH --max_new_tokens $MAX_NEW_TOKENS --output_json "$CAL_JSON" --output_md "$CAL_MD"


In [ ]:
cal = json.loads(Path(CAL_JSON).read_text(encoding='utf-8'))
print('metadata:')
print(json.dumps(cal['metadata'], indent=2, ensure_ascii=False))
print()
print('best searched candidate:')
print(json.dumps(cal['search']['best_candidate'], indent=2, ensure_ascii=False))
print()
print('top search candidates:')
print(json.dumps(cal['search']['top_candidates'][:5], indent=2, ensure_ascii=False))
print()
print('threshold candidates summary:')
print(json.dumps(cal['calibration']['threshold_candidates'], indent=2, ensure_ascii=False))
print()
for cluster, summary in cal['calibration']['by_cluster_clean_base'].items():
    print(cluster, json.dumps(summary, ensure_ascii=False))
print()
print('policy simulation (all_cases):')
print(json.dumps(cal['policy_simulation']['all_cases'], indent=2, ensure_ascii=False))
print()
print('report head:')
print(Path(CAL_MD).read_text(encoding='utf-8')[:8000])


## Optional legacy diagnostics

Ниже — только для дополнительной диагностики. Для основного сбора данных **необязательно**.

- `scripts/run_qwen_anchor_geometry_probe.py` — отдельный старый geometry report;
- `scripts/calibrate_qwen_anchor_thresholds.py` — legacy detector threshold sweep, не crystallization search.


In [ ]:
# Optional artifact download
# from google.colab import files
# for artifact in [CAL_JSON, CAL_MD]:
#     files.download(artifact)
